# Stock Market Prediction Using Machine Learning and Ensemble Learning

## Imports

In [70]:
import pandas as pd
import numpy as np
import seaborn as sns
import yfinance as yf
import matplotlib.pyplot as plt
import joblib

from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split,TimeSeriesSplit,KFold,cross_validate,RandomizedSearchCV)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,r2_score)

sns.set_theme()

## Data Processing


In [71]:
# Ticker = input(print('Enter Ticker: '))
Ticker ='MSFT'

df = yf.download(Ticker,'2020-01-01')

[*********************100%***********************]  1 of 1 completed


In [72]:
df.head()

Price,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,
2020-01-02,151.829559,151.933540,149.664893,150.090263,22622100
2020-01-03,149.938995,151.196208,149.409646,149.655425,21116200
2020-01-06,150.326569,150.392745,147.944480,148.483292,20813700
2020-01-07,148.955917,150.931532,148.710152,150.600695,21634100
2020-01-08,151.328568,151.999717,149.305686,150.232049,27746500


In [73]:
df['Target'] = df['Close'].shift(-1)
df.head()

Price,Close,High,Low,Open,Volume,Target
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,
Date,,,,,,
2020-01-02,151.829559,151.933540,149.664893,150.090263,22622100,149.938995
2020-01-03,149.938995,151.196208,149.409646,149.655425,21116200,150.326569
2020-01-06,150.326569,150.392745,147.944480,148.483292,20813700,148.955917
2020-01-07,148.955917,150.931532,148.710152,150.600695,21634100,151.328568
2020-01-08,151.328568,151.999717,149.305686,150.232049,27746500,153.219131


In [74]:

df['MA_10'] = df['Close'].rolling(10).mean()
df['MA_50'] = df['Close'].rolling(50).mean()


df['Volatility'] = df['Close'].rolling(10).std()


df['Daily_Return'] = df['Close'].pct_change()

In [75]:
df.dropna(axis=0,inplace=True)

In [76]:
df.describe()

Price,Close,High,Low,Open,Volume,Target,MA_10,MA_50,Volatility,Daily_Return
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,,,,,
count,1611.000000,1611.000000,1611.000000,1611.000000,1.611000e+03,1611.000000,1611.000000,1611.000000,1611.000000,1611.000000
mean,328.667522,331.817068,325.264028,328.598099,2.810566e+07,328.879926,327.692871,324.704804,6.447199,0.001007
std,97.899169,98.437069,97.343280,97.985906,1.292116e+07,97.883678,97.680904,98.041844,4.407729,0.018750
min,128.358337,133.239789,125.609564,129.865418,5.855900e+06,128.358337,135.692831,153.334287,0.841681,-0.147390
25%,244.532112,246.754740,241.438225,243.961245,1.994990e+07,244.636513,244.437039,241.105911,3.866775,-0.008082
50%,319.236481,322.457631,316.378243,319.305398,2.513880e+07,319.642670,318.918948,313.413365,5.551123,0.000846
75%,410.134247,413.558983,406.171606,410.429852,3.266555e+07,410.168839,411.436836,409.659534,7.654940,0.010485
max,538.658569,551.048474,537.366763,550.830186,1.862016e+08,538.658569,522.501874,511.207400,49.547926,0.155067


In [77]:
df.isnull().sum()


Price         Ticker
Close         MSFT      0
High          MSFT      0
Low           MSFT      0
Open          MSFT      0
Volume        MSFT      0
Target                  0
MA_10                   0
MA_50                   0
Volatility              0
Daily_Return            0
dtype: int64

## Feature Selection

In [78]:
target = df[['Target']]
features = df[['Close','Volume','High','Low','Open',
         'MA_10','MA_50',
         'Volatility']]

## Train Test Split

In [79]:
x_train,x_test,y_train,y_test = train_test_split(features,target,test_size = 0.2,shuffle = False)

## Standardization


In [80]:
scaler = StandardScaler()

# Scaling the features
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [81]:
# saving the scaler
joblib.dump(scaler, "../model/scaler.pkl")

['../model/scaler.pkl']

## TimeSeries Split 

In [82]:
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(x_train_scaled), 1):

    x_train_cv = x_train_scaled[train_idx]
    x_val_cv = x_train_scaled[val_idx]

    y_train_cv = y_train.iloc[train_idx]
    y_val_cv = y_train.iloc[val_idx]


## Cross Validation

In [83]:
scoring = {
    'R2':'r2',
    'MAE':'neg_mean_absolute_error',
    'RMSE':'neg_root_mean_squared_error'
}

def cross_validation(model,x,y):
    scores = cross_validate(
        model,
        x,
        y,
        cv = tscv,
        scoring = scoring
    )

    print(f"Average R2 : {scores["test_R2"].mean()}")
    print(f"Average MAE : {-scores["test_MAE"].mean()}")
    print(f"Average RMSE : {-scores["test_RMSE"].mean()}")


## Linear Regression

In [84]:
lr = LinearRegression()

cross_validation(lr,x_train_cv,y_train_cv)

Average R2 : 0.9451541092415141
Average MAE : 3.7964229964679177
Average RMSE : 4.827765071791068


##  Random Forest

In [85]:
rf = RandomForestRegressor()

cross_validation(rf,x_train_cv,y_train_cv)

C:\Users\sankp\anaconda3\envs\TF_env\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\sankp\anaconda3\envs\TF_env\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\sankp\anaconda3\envs\TF_env\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\sankp\anaconda3\envs\TF_env\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. P

Average R2 : -0.5509664421997764
Average MAE : 21.129593048948152
Average RMSE : 25.667746064535333


## XG Boost Regressor

In [86]:
xgb = XGBRegressor()

cross_validation(xgb,x_train_cv,y_train_cv)

Average R2 : -0.8935311198234558
Average MAE : 23.783495330810545
Average RMSE : 28.27630500793457
